# Federated MedMNIST 3D con Flower

Questo notebook mantiene il walkthrough di dataset e modello, ma usa il **Message API** moderno di Flower per l'esecuzione federata. La parte Flower vera e propria vive in `real_fed/fl_medmnist3d/fl_medmnist3d/` e viene lanciata con `flwr run`, così gli studenti vedono gli stessi comandi che userebbero da terminale.


## 1 — Installazione dipendenze

In [1]:
# From a terminal, install the workshop dependencies once:
# python -m pip install -r requirements-workshop.txt

## 2 — Import e configurazione globale

In [2]:
import os
import warnings
from pathlib import Path
from typing import List, Tuple

warnings.filterwarnings("ignore")
os.environ["FLWR_TELEMETRY_ENABLED"] = "0"
os.environ.setdefault("MEDMNIST_ROOT", str(Path(".data/medmnist").resolve()))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import medmnist
from medmnist import INFO
from medmnist.evaluator import getACC

import importlib
import workshop_helpers

importlib.reload(workshop_helpers)
from workshop_helpers import bin_dir, preflight, run_stream

ctx = preflight("real_fed/fl_medmnist3d")
flwr_bin_prefix = bin_dir(ctx)

print(f"PyTorch  : {torch.__version__}")
print(f"MedMNIST : {medmnist.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")

Python executable : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/python
Python version    : 3.13.5
Notebook cwd      : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos
Flower app        : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/real_fed/fl_medmnist3d
flwr binary       : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr


flwr            : 1.30.0 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/flwr
numpy           : 2.4.6 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/numpy
PyTorch  : 2.12.0
MedMNIST : 3.0.2
CUDA     : False


In [3]:
# ── Iperparametri globali ──────────────────────────────────────────────────────
DATA_FLAG     = "synapsemnist3d"   # dataset MedMNIST 3D
NUM_CLIENTS   = 2                  # numero di client federati (SuperNode)
NUM_ROUNDS    = 3                  # round di aggregazione FedAvg
LOCAL_EPOCHS  = 1                  # epoche locali per round
BATCH_SIZE    = 64
LR            = 0.001

# ── Info dataset ──────────────────────────────────────────────────────────────
info      = INFO[DATA_FLAG]
TASK      = info["task"]
N_CHANNELS = info["n_channels"]
N_CLASSES  = len(info["label"])
DataClass  = getattr(medmnist, info["python_class"])

print(f"Dataset  : {DATA_FLAG}")
print(f"Task     : {TASK}")
print(f"Channels : {N_CHANNELS}  |  Classes: {N_CLASSES}")
print(info)

Dataset  : synapsemnist3d
Task     : binary-class
Channels : 1  |  Classes: 2
{'python_class': 'SynapseMNIST3D', 'description': 'The SynapseMNIST3D is a new 3D volume dataset to classify whether a synapse is excitatory or inhibitory. It uses a 3D image volume of an adult rat acquired by a multi-beam scanning electron microscope. The original data is of the size 100×100×100um^3 and the resolution 8×8×30nm^3, where a (30um)^3 sub-volume was used in the MitoEM dataset with dense 3D mitochondria instance segmentation labels. Three neuroscience experts segment a pyramidal neuron within the whole volume and proofread all the synapses on this neuron with excitatory/inhibitory labels. For each labeled synaptic location, we crop a 3D volume of 1024×1024×1024nm^3 and resize it into 28×28×28 voxels. Finally, the dataset is randomly split with a ratio of 7:1:2 into training, validation and test set.', 'url': 'https://zenodo.org/records/10519652/files/synapsemnist3d.npz?download=1', 'MD5': '1235b78

## 3 — Dataset e Data Loading

Sostituisce `DataInterface` + `TransformedDataset` di OpenFL con classi standard PyTorch.  
Il training set viene suddiviso uniformemente tra i `NUM_CLIENTS` client.

In [4]:
class Transform3D:
    """Normalizza un volume 3D e applica opzionalmente uno scaling."""

    def __init__(self, mul=None):
        self.mul = mul

    def __call__(self, voxel: np.ndarray) -> np.ndarray:
        if self.mul == "random":
            voxel = voxel * np.random.uniform(0, 1)
        elif self.mul is not None:
            voxel = voxel * float(self.mul)
        return voxel.astype(np.float32)


class MedMNIST3DDataset(Dataset):
    """Wrapper per dataset MedMNIST 3D con normalizzazione e transform.

    MedMNIST 3D restituisce immagini con shape (C, D, H, W) dove C=n_channels.
    Si normalizza in [0,1] senza aggiungere dimensioni extra.
    """

    def __init__(self, medmnist_ds, transform=None):
        self.dataset   = medmnist_ds
        self.transform = transform

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img, label = self.dataset[index]
        # img ha già shape (C, D, H, W) — normalizza senza aggiungere assi
        img = img.astype(np.float32) / 255.0
        if self.transform is not None:
            img = self.transform(img)
        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(label.astype(int)),   # shape (1,)
        )

In [5]:
def load_data(
    partition_id: int,
    num_partitions: int,
    train_bs: int = BATCH_SIZE,
    val_bs: int = 512,
) -> Tuple[DataLoader, DataLoader]:
    """Scarica MedMNIST e restituisce i loader per il client `partition_id`.

    Il training set viene diviso in `num_partitions` slice contigue.
    Il validation set rimane completo per ogni client (valutazione globale).
    """
    train_transform = Transform3D()
    eval_transform  = Transform3D()

    raw_train = DataClass(split="train", download=True)
    raw_val   = DataClass(split="val",   download=True)

    # Partizionamento uniforme del training set
    n_total   = len(raw_train)
    per_slice = n_total // num_partitions
    start     = partition_id * per_slice
    end       = start + per_slice if partition_id < num_partitions - 1 else n_total
    indices   = list(range(start, end))

    train_ds  = MedMNIST3DDataset(raw_train, transform=train_transform)
    val_ds    = MedMNIST3DDataset(raw_val,   transform=eval_transform)

    trainloader = DataLoader(Subset(train_ds, indices), batch_size=train_bs, shuffle=True,  num_workers=0)
    valloader   = DataLoader(val_ds,                    batch_size=val_bs,   shuffle=False, num_workers=0)

    return trainloader, valloader


# ── Verifica rapida ──────────────────────────────────────────────────────────
tl, vl = load_data(partition_id=0, num_partitions=NUM_CLIENTS)
sample, target = next(iter(tl))
print(f"Train batches: {len(tl)}  |  Batch shape: {sample.shape}  |  Label shape: {target.shape}")
print(f"Val   batches: {len(vl)}")

Train batches: 10  |  Batch shape: torch.Size([64, 1, 28, 28, 28])  |  Label shape: torch.Size([64, 1])
Val   batches: 1


## 4 — Definizione del modello

`Net3D` è una CNN scritta **direttamente in 3D** con layer `Conv3d`, `BatchNorm3d` e `MaxPool3d`.  
Questa è la versione equivalente alla `Net` originale dopo la conversione con ACSConv,  
ma senza la dipendenza esterna — più robusta con versioni recenti di PyTorch.

Dimensioni spaziali su input `28×28×28`:
- layer1 (Conv3d 3): `26×26×26`
- layer2 (Conv3d 3 + MaxPool3d 2): `24×24×24` → `12×12×12`
- layer3 (Conv3d 3): `10×10×10`
- layer4 (Conv3d 3): `8×8×8`
- layer5 (Conv3d 3 p=1 + MaxPool3d 2): `8×8×8` → `4×4×4`
- fc: `64 × 4 × 4 × 4 = 4096 → 128 → num_classes`

In [6]:
class Net3D(nn.Module):
    """CNN 3D nativa — equivalente alla Net originale dopo conversione ACSConv."""

    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Conv3d(in_channels, 16, 3), nn.BatchNorm3d(16), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Conv3d(16, 16, 3),          nn.BatchNorm3d(16), nn.ReLU(), nn.MaxPool3d(2, 2))
        self.layer3 = nn.Sequential(nn.Conv3d(16, 64, 3),          nn.BatchNorm3d(64), nn.ReLU())
        self.layer4 = nn.Sequential(nn.Conv3d(64, 64, 3),          nn.BatchNorm3d(64), nn.ReLU())
        self.layer5 = nn.Sequential(nn.Conv3d(64, 64, 3, padding=1), nn.BatchNorm3d(64), nn.ReLU(), nn.MaxPool3d(2, 2))
        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, 128),             nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.layer5(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


def build_model() -> nn.Module:
    return Net3D(in_channels=N_CHANNELS, num_classes=N_CLASSES)


# ── Verifica architettura ────────────────────────────────────────────────────
model_test = build_model()
dummy = torch.zeros(2, N_CHANNELS, 28, 28, 28)   # batch=2, C=1, D=28, H=28, W=28
out   = model_test(dummy)
print(f"Output shape: {out.shape}  (atteso: [2, {N_CLASSES}])")
n_params = sum(p.numel() for p in model_test.parameters())
print(f"Parametri totali: {n_params:,}")
del model_test, dummy, out

Output shape: torch.Size([2, 2])  (atteso: [2, 2])
Parametri totali: 798,034


## 5 — Flower ClientApp con Message API

La vecchia versione del notebook definiva un `NumPyClient` direttamente in Jupyter. La versione aggiornata usa il Message API: il `ClientApp` riceve un `Message`, legge i pesi da un `ArrayRecord`, addestra/valuta localmente e risponde con `ArrayRecord` + `MetricRecord`.

In [7]:
print(Path("real_fed/fl_medmnist3d/fl_medmnist3d/client_app.py").read_text())

"""ClientApp: MedMNIST 3D training using Flower's Message API."""

from __future__ import annotations

import torch
from flwr.app import ArrayRecord, Context, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp

from fl_medmnist3d.task import Net3D, evaluate, load_data, train


app = ClientApp()


def _partition_id(context: Context, num_partitions: int) -> int:
    """Return a bounded partition id for local simulation and SuperNode runs."""
    configured = context.node_config.get("partition-id")
    if configured is not None:
        return int(configured)
    return int(context.node_id) % num_partitions


@app.train()
def train_message(msg: Message, context: Context) -> Message:
    """Train locally and return updated arrays plus weighted metrics."""
    config = msg.content.get("config", {})
    num_partitions = int(context.run_config.get("num-partitions", 2))
    local_epochs = int(config.get("local-epochs", context.run_config.get("local-epochs", 1)))
    lr = fl

## 6 — ServerApp e FedAvg con Message API

Il server usa `flwr.serverapp.strategy.FedAvg`. La strategia invia `ArrayRecord` e `ConfigRecord` ai client, poi aggrega le metriche usando `num-examples`.

In [8]:
print(Path("real_fed/fl_medmnist3d/fl_medmnist3d/server_app.py").read_text())

"""ServerApp: Message API FedAvg aggregation for MedMNIST 3D."""

from __future__ import annotations

from flwr.app import ArrayRecord, ConfigRecord, Context
from flwr.serverapp import Grid, ServerApp
from flwr.serverapp.strategy import FedAvg

from fl_medmnist3d.task import Net3D


app = ServerApp()


@app.main()
def main(grid: Grid, context: Context) -> None:
    """Run FedAvg over Message API ClientApps."""
    num_rounds = int(context.run_config.get("num-server-rounds", 3))
    min_clients = int(context.run_config.get("min-clients", 2))
    local_epochs = int(context.run_config.get("local-epochs", 1))
    lr = float(context.run_config.get("learning-rate", 0.001))

    strategy = FedAvg(
        fraction_train=1.0,
        fraction_evaluate=1.0,
        min_train_nodes=min_clients,
        min_evaluate_nodes=min_clients,
        min_available_nodes=min_clients,
    )

    initial_arrays = ArrayRecord(torch_state_dict=Net3D().state_dict())
    train_config = ConfigRecord({"local-epoc

## 7 — Avvio della simulazione federata

Questo comando lancia la Flower App. È lo stesso comando che uno studente può copiare nel terminale dalla root del repository.

In [9]:
command = r"""
flwr run real_fed/fl_medmnist3d local \
--run-config num-server-rounds=3 \
--run-config min-clients=2 \
--run-config num-partitions=2 \
--run-config local-epochs=1 \
--run-config learning-rate=0.001 \
--federation-config num-supernodes=2 \
--stream
""".strip()

rc = run_stream(flwr_bin_prefix + command, cwd=ctx.root)
if rc != 0:
    raise RuntimeError(f"MedMNIST Message API simulation failed with exit code {rc}")

$ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr run real_fed/fl_medmnist3d local \
  --run-config num-server-rounds=3 \
  --run-config min-clients=2 \
  --run-config num-partitions=2 \
  --run-config local-epochs=1 \
  --run-config learning-rate=0.001 \
  --federation-config num-supernodes=2 \
  --stream


🎊 Successfully started run 9857701588056124163


INFO :      Starting logstream for run_id `9857701588056124163`


INFO :      Starting Flower Simulation


INFO :      Starting FedAvg strategy:


...

  2: {'accuracy': '7.2881e-01', 'eval_loss': '5.8555e-01'},


  3: {'accuracy': '7.2881e-01', 'eval_loss': '6.0084e-01'}}


ServerApp-side Evaluate Metrics:


{}


INFO :      


## 8 — Come leggere l'output Flower

Nell'output di `flwr run`, gli studenti dovrebbero seguire questi punti:

- `Starting FedAvg strategy`: conferma numero di round, record keys e criterio di pesatura.
- `configure_train`: quanti nodi/client sono stati campionati per il training.
- `aggregate_train`: metriche aggregate lato training, ad esempio `train_loss`.
- `configure_evaluate` e `aggregate_evaluate`: valutazione federata e metriche come `accuracy` ed `eval_loss`.
- `Final results`: riepilogo finale degli `ArrayRecord` globali e delle metriche aggregate.